In [3]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer
import torch
import textstat

In [4]:
# Load sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


In [5]:
def sentiment_accuracy(predictions, references):
    correct = 0
    for pred, ref in zip(predictions, references):
        pred_sentiment = sentiment_analyzer(pred)[0]['label']
        ref_sentiment = sentiment_analyzer(ref)[0]['label']
        if pred_sentiment == ref_sentiment:
            correct += 1
    return correct / len(predictions)

# BLEU Score Evaluation
def bleu_score(predictions, references):
    scores = [sentence_bleu([ref.split()], pred.split()) for pred, ref in zip(predictions, references)]
    return sum(scores) / len(scores)

# Perplexity Evaluation
def calculate_perplexity(predictions):
    model_name = "gpt2"
    model = GPT2LMHeadModel.from_pretrained(model_name)
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model.eval()
    perplexities = []
    
    for sentence in predictions:
        encodings = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            loss = model(**encodings, labels=encodings["input_ids"]).loss
        perplexity = torch.exp(loss).item()
        perplexities.append(perplexity)
    
    return sum(perplexities) / len(perplexities)

In [8]:
# Example usage
predictions = ["I think that we should all go vegan! Meat is bad for the earth and for our bodies. First, it takes a lot of water to grow food for animals. Like, did you know it takes 1,000 gallons of water to make just one pound of beef? That's crazy! We could use that water for people who don't have any. And it's not just water, it's also space. Cows take up so much room, and they make so much poop. It's bad for the soil, and it's bad for the air. We should just eat plants instead. They're good for us, and they don't hurt the earth as much. ALUs, plants are yummy! Have you tried tofu? It's so good! And you can put it in anything. My favorite is tofu stir fry with broccoli and carrots. It's so good!"]
references = ["I believe a shift towards veganism is something we should seriously consider. Consuming meat has negative consequences for both the planet and our health. For one, animal agriculture demands an immense amount of water to produce feed. Just consider the fact that producing a single pound of beef requires approximately 1,000 gallons of water. Isn't that astonishing? That water could be used to help individuals facing water scarcity. Furthermore, the land requirements for raising livestock are substantial. Cattle require a significant amount of space, and their waste contributes to soil and air pollution. A plant-based diet, on the other hand, benefits both our health and the environment. Plus, plants can be delicious! Have you ever tasted tofu? It's incredibly versatile and can be incorporated into countless dishes. I personally love tofu stir-fry with broccoli and carrots; it's absolutely delicious."]

sent_acc = sentiment_accuracy(predictions, references)
bleu = bleu_score(predictions, references)
ppl = calculate_perplexity(predictions)

print(f"Sentiment Accuracy: {sent_acc:.2f}")
print(f"BLEU Score: {bleu:.2f}")
print(f"Perplexity: {ppl:.2f}")

Sentiment Accuracy: 1.00
BLEU Score: 0.00
Perplexity: 13.87
